# Sistema de Recomendación - E-commerce

## Objetivo
Incrementar el ticket promedio mediante recomendaciones personalizadas y cross-selling basado en comportamiento de compra.

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [30]:
#Carga del Dataset
import os

BASE_PATH = "../data/raw"

products = pd.read_csv(os.path.join(BASE_PATH, "olist_products_dataset.csv"))
orders = pd.read_csv(os.path.join(BASE_PATH, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(BASE_PATH, "olist_order_items_dataset.csv"))
reviews = pd.read_csv(os.path.join(BASE_PATH, "olist_order_reviews_dataset.csv"))
payments = pd.read_csv(os.path.join(BASE_PATH, "olist_order_payments_dataset.csv"))
customers = pd.read_csv(os.path.join(BASE_PATH, "olist_customers_dataset.csv"))
translation = pd.read_csv(os.path.join(BASE_PATH, "product_category_name_translation.csv"))

Se priorizarán las tablas directamente relacionadas con el comportamiento transaccional, el catálogo de productos, las reseñas, los pagos y la información de clientes, complementadas con la tabla de traducción de categorías para mejorar la interpretación del negocio. Las tablas de vendedores y geolocalización serán consideradas de manera secundaria, dado que no son críticas para el MVP inicial del sistema de recomendación.

In [31]:
# Diccionario de Tablas
tablas = {
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "reviews": reviews,
    "payments": payments,
    "customers": customers,
    "translation": translation
}

In [32]:
# Reconocimiento inicial de los datos
for nombre, df in tablas.items():
    print(f"\n--- {nombre.upper()} ---")
    print("Filas y columnas:", df.shape)
    print(df.head(3))


--- PRODUCTS ---
Filas y columnas: (32951, 9)
                         product_id product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5            perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                 artes   
2  96bd76ec8810374ed1b65e291975717f         esporte_lazer   

   product_name_lenght  product_description_lenght  product_photos_qty  \
0                 40.0                       287.0                 1.0   
1                 44.0                       276.0                 1.0   
2                 46.0                       250.0                 1.0   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  
0             225.0               16.0               10.0              14.0  
1            1000.0               30.0               18.0              20.0  
2             154.0               18.0                9.0              15.0  

--- ORDERS ---
Filas y columnas: (99441, 8)
                           order_id                   

- La tabla products contiene información de 32,951 productos con 9 variables asociadas a sus características físicas y descriptivas.
Se identifican variables relevantes como la categoría del producto, longitud del nombre y descripción, cantidad de fotos y dimensiones físicas (peso, alto, ancho y largo).

    Esto permite inferir que el dataset no solo contiene información comercial, sino también atributos que pueden influir en la experiencia del cliente, como la calidad de la descripción o la presentación visual del producto.

- La tabla orders contiene 99,441 registros correspondientes a pedidos realizados por los clientes.
Cada registro representa una orden única, identificada mediante order_id, y asociada a un cliente mediante customer_id.

    Esta tabla es fundamental dentro del modelo de datos, ya que permite analizar el comportamiento de compra, frecuencia de pedidos y estados de las órdenes.

    Su volumen es considerablemente mayor que el de productos, lo que sugiere que múltiples pedidos pueden estar asociados a los mismos productos, lo cual será relevante al momento de realizar análisis relacionales.

In [33]:
# Comprensión de variables
for nombre, df in tablas.items():
    print(f"\n--- COLUMNAS DE {nombre.upper()} ---")
    print(df.columns.tolist())


--- COLUMNAS DE PRODUCTS ---
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

--- COLUMNAS DE ORDERS ---
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

--- COLUMNAS DE ORDER_ITEMS ---
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

--- COLUMNAS DE REVIEWS ---
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

--- COLUMNAS DE PAYMENTS ---
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

--- COLUMNAS DE CUSTOMERS ---
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'custom

- La tabla products contiene variables relacionadas con las características físicas y descriptivas de los productos.
Se identifican atributos como categoría, longitud del nombre y descripción, cantidad de fotos y dimensiones físicas (peso, alto, ancho y largo). Estas variables son relevantes para analizar la calidad de la información del producto y su posible influencia en las decisiones de compra.

- La tabla orders contiene información transaccional de los pedidos, incluyendo su estado y diferentes marcas de tiempo asociadas al proceso logístico.
Se identifican variables clave como:

        fecha de compra,

        aprobación,

        envío,

        entrega al cliente,

        y fecha estimada de entrega.

- La tabla order_items representa el detalle de cada producto dentro de un pedido.
Permite relacionar pedidos con productos y vendedores, incluyendo información sobre precio y costo de envío.

- La tabla reviews contiene las valoraciones realizadas por los clientes sobre los pedidos.
Incluye tanto variables cuantitativas (calificación) como cualitativas (comentarios).

- La tabla payments contiene información sobre los métodos de pago utilizados en cada pedido.
Incluye el tipo de pago, número de cuotas y valor pagado.
Esta tabla permite analizar preferencias de pago, comportamiento financiero de los clientes y distribución de ingresos.

- La tabla customers contiene información geográfica de los clientes.
Incluye ciudad, estado y código postal, lo que permite realizar análisis de distribución geográfica, segmentación de clientes y estudios de mercado por región.


In [34]:
# tipos de datos y calidad
for nombre, df in tablas.items():
    print(f"\n--- INFO DE {nombre.upper()} ---")
    print(df.info())


--- INFO DE PRODUCTS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB
None

--- INFO DE ORDERS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count

En general, se observa que los datos se encuentran en formatos adecuados; sin embargo, existen oportunidades de mejora en la tipificación de variables. 

- La tabla products presenta valores faltantes en varias de sus variables.

    Se identifican aproximadamente 610 registros sin información en variables clave, como la categoría del producto, longitud del nombre, descripción y cantidad de fotos.

    Esto puede afectar análisis relacionados con segmentación de productos o calidad de información.

    Por otro lado, las variables físicas (peso y dimensiones) presentan un número mínimo de valores faltantes (solo 2 registros), lo que indica una alta completitud en estos atributos.

    En cuanto a los tipos de datos, se observa que las variables numéricas están correctamente definidas como float64, aunque algunas de ellas podrían considerarse discretas (por ejemplo, cantidad de fotos), lo cual se evaluará en etapas posteriores.

- La tabla orders contiene múltiples variables de tipo fecha que describen el ciclo de vida de un pedido.

    Se identifican valores faltantes en variables relacionadas con la aprobación y entrega de pedidos, lo cual es esperado, ya que no todos los pedidos han sido completados o entregados.

    Estos valores nulos no necesariamente representan errores, sino estados del proceso logístico (por ejemplo, pedidos en curso o cancelados).

    Los tipos de datos corresponden principalmente a cadenas de texto (str), lo que sugiere la necesidad de convertir las variables de fecha a formato datetime para facilitar su análisis.

In [35]:
# Exploración estructural del dataset
for nombre, df in tablas.items():
    print(f"\n--- {nombre.upper()} ---")
    print("Dimensiones:", df.shape)
    print("Columnas:", df.columns.tolist())
    display(df.head(3))


--- PRODUCTS ---
Dimensiones: (32951, 9)
Columnas: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0



--- ORDERS ---
Dimensiones: (99441, 8)
Columnas: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



--- ORDER_ITEMS ---
Dimensiones: (112650, 7)
Columnas: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


...

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71



--- CUSTOMERS ---
Dimensiones: (99441, 5)
Columnas: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



--- TRANSLATION ---
Dimensiones: (71, 2)
Columnas: ['product_category_name', 'product_category_name_english']


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


Se identifican entidades principales como productos, pedidos, clientes, pagos y reseñas, lo que permite analizar el ciclo completo de compra desde la selección del producto hasta la evaluación del servicio por parte del cliente.

La diversidad de tablas y su volumen de datos permiten realizar análisis descriptivos, predictivos y de comportamiento del cliente.

A partir de la estructura de las tablas, se identifican las siguientes relaciones:

    - orders se relaciona con customers mediante customer_id

    - order_items conecta pedidos con productos (order_id, product_id)

    - products se relaciona con translation mediante product_category_name

    - reviews y payments se vinculan con orders mediante order_id

Esto configura un modelo relacional donde:

    - orders funciona como tabla central de hechos

    - order_items como tabla de detalle

    - products, customers y translation como dimensiones

El número de pedidos coincide con el número de clientes (99,441), lo que sugiere un identificador único por compra

Existen más registros en order_items y payments, lo que indica relaciones uno a muchos

La tabla de productos es más pequeña, lo que indica reutilización de productos en múltiples pedidos

La estructura permite análisis de comportamiento, logística y satisfacción del cliente

In [36]:
# columnas con nulos
for nombre, df in tablas.items():
    print(f"\n--- NULOS EN {nombre.upper()} ---")
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0].sort_values(ascending=False)
    print(nulos if not nulos.empty else "No hay nulos")


--- NULOS EN PRODUCTS ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- NULOS EN ORDERS ---
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

--- NULOS EN ORDER_ITEMS ---
No hay nulos

--- NULOS EN REVIEWS ---
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- NULOS EN PAYMENTS ---
No hay nulos

--- NULOS EN CUSTOMERS ---
No hay nulos

--- NULOS EN TRANSLATION ---
No hay nulos


El análisis de valores faltantes permitió identificar que el dataset presenta una buena calidad general.

La mayoría de las tablas no contienen valores nulos o presentan porcentajes mínimos, lo que garantiza la confiabilidad de los análisis.

- La tabla products presenta valores nulos en variables descriptivas clave, con aproximadamente 610 registros incompletos en categoría, nombre, descripción y número de fotos. Esto sugiere posibles problemas en el registro de información del catálogo de productos.
Sin embargo, el porcentaje de valores faltantes es relativamente bajo en comparación con el total de registros (~1.8%), por lo que no representa un problema crítico.

- La tabla orders presenta valores nulos en variables relacionadas con el proceso de entrega y aprobación de pedidos.
Estos valores faltantes no representan errores en los datos, sino que corresponden a pedidos que no han sido completados, entregados o aprobados. Por lo tanto, estos nulos tienen un significado operativo y deben ser interpretados como parte del ciclo de vida del pedido.

- La tabla order_items no presenta valores faltantes, lo que indica una alta calidad en el registro de los detalles de los pedidos.

- La tabla reviews presenta una gran cantidad de valores nulos en las variables de texto asociadas a comentarios.
Esto indica que la mayoría de los usuarios no dejan comentarios escritos, aunque sí proporcionan una calificación numérica. 

- La tabla payments no presenta valores faltantes, lo que garantiza la integridad de la información financiera asociada a los pedidos.

- La tabla customers presenta información completa, lo que permite realizar análisis geográficos sin necesidad de procesos adicionales de limpieza.

In [37]:
# registros duplicados exactos
for nombre, df in tablas.items():
    print(f"{nombre}: duplicados totales = {df.duplicated().sum()}")

products: duplicados totales = 0
orders: duplicados totales = 0
order_items: duplicados totales = 0
reviews: duplicados totales = 0
payments: duplicados totales = 0
customers: duplicados totales = 0
translation: duplicados totales = 0


Se realizó la verificación de registros duplicados en todas las tablas del dataset, encontrándose que no existen duplicados exactos en ninguna de ellas.

Este resultado indica una alta calidad en la integridad de los datos, ya que cada registro es único dentro de su respectiva tabla.

In [38]:
# Validación de claves primarias
print("Duplicados product_id en products:", products["product_id"].duplicated().sum())
print("Duplicados order_id en orders:", orders["order_id"].duplicated().sum())
print("Duplicados customer_id en customers:", customers["customer_id"].duplicated().sum())

Duplicados product_id en products: 0
Duplicados order_id en orders: 0
Duplicados customer_id en customers: 0


La validación de claves primarias confirma que el dataset presenta una estructura consistente, sin duplicados en los identificadores principales.

Esto asegura la integridad de los datos y permite realizar análisis relacionales sin riesgo de inconsistencias.

En conjunto con los análisis de valores nulos y duplicados generales, se concluye que el dataset tiene una alta calidad y es adecuado para procesos de análisis exploratorio avanzado y modelado.

In [39]:
# combinación de tabla products y translation
products = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


Se realizó un proceso de integración de datos entre la tabla de productos y la tabla de traducción de categorías, utilizando la variable product_category_name como llave de unión.

La operación se efectuó mediante un left join, con el fin de conservar la totalidad de los registros de productos e incorporar, cuando existiera correspondencia

In [ ]:
#Diccionario de grupos para cada categoría
mapa_grupos = {
    # HOME
    "bed_bath_table": "Hogar y Decoración",
    "furniture_decor": "Hogar y Decoración",
    "housewares": "Hogar y Decoración",
    "home_comfort": "Hogar y Decoración",
    "living_room_furniture": "Hogar y Decoración",
    "garden_tools": "Hogar y Decoración",

    # PERSONAL CARE / LIFESTYLE
    "health_beauty": "Cuidado Personal",
    "perfumery": "Cuidado Personal",
    "fashion_bags_accessories": "Cuidado Personal",
    "fashion_shoes": "Cuidado Personal",
    "watches_gifts": "Cuidado Personal",

    # TECH
    "computers_accessories": "Tecnología y entretenimiento",
    "telephony": "Tecnología y entretenimiento",
    "electronics": "Tecnología y entretenimiento",
    "consoles_games": "Tecnología y entretenimiento",
    "audio": "Tecnología y entretenimiento",

    # LIFESTYLE / FAMILY / PET
    "baby": "Familia, mascotas y recreacion",
    "toys": "Familia, mascotas y recreacion",
    "pet_shop": "Familia, mascotas y recreacion",
    "stationery": "Familia, mascotas y recreacion",
    "sports_leisure": "Familia, mascotas y recreacion"
}

In [48]:
# convertir fechas a formato datetime
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

In [44]:
# Creamos Columna de grupos
products["grupo_categoria"] = (
    products["product_category_name_english"]
    .map(mapa_grupos)
    .fillna("other")
)

In [45]:
products["grupo_categoria"].value_counts()

grupo_categoria
Hogar y Decoración                8774
other                             8084
Familia, mascotas y recreacion    6765
Cuidado Personal                  5663
Tecnología y entretenimiento      3665
Name: count, dtype: int64

In [46]:
# categorías sin grupo asignado
products[products["grupo_categoria"] == "other"]["product_category_name_english"].unique()

array(['art', 'musical_instruments', 'cool_stuff', 'home_appliances',
       'construction_tools_safety', 'luggage_accessories',
       'office_furniture', 'auto', 'computers', 'home_construction',
       'construction_tools_construction', 'small_appliances',
       'agro_industry_and_commerce', nan, 'furniture_living_room',
       'signaling_and_security', 'air_conditioning',
       'books_general_interest', 'costruction_tools_tools',
       'fashion_underwear_beach', 'fashion_male_clothing',
       'kitchen_dining_laundry_garden_furniture',
       'industry_commerce_and_business', 'fixed_telephony',
       'construction_tools_lights', 'books_technical',
       'home_appliances_2', 'party_supplies', 'drinks', 'market_place',
       'la_cuisine', 'costruction_tools_garden', 'fashio_female_clothing',
       'home_confort', 'food_drink', 'music', 'food',
       'tablets_printing_image', 'books_imported',
       'small_appliances_home_oven_and_coffee', 'fashion_sport',
       'christmas_s

In [50]:
# DATASET FINAL

df_model = (
    order_items
    .merge(
        orders[["order_id", "customer_id", "order_purchase_timestamp", "order_status"]],
        on="order_id",
        how="left"
    )
    .merge(
        products[["product_id", "product_category_name_english", "grupo_categoria"]],
        on="product_id",
        how="left"
    )
    .merge(
        reviews[["order_id", "review_score"]],
        on="order_id",
        how="left"
    )
    .merge(
        payments[["order_id", "payment_value"]],
        on="order_id",
        how="left"
    )
)

In [51]:
# Limpiar valores criticos
df_model = df_model.dropna(subset=[
    "order_id",
    "customer_id",
    "product_id",
    "price"
])

In [52]:
# ordenes validas por variables temporales
df_model["year"] = df_model["order_purchase_timestamp"].dt.year
df_model["month"] = df_model["order_purchase_timestamp"].dt.month
df_model["day_of_week"] = df_model["order_purchase_timestamp"].dt.dayofweek


In [55]:
# por hora del dia
df_model["hour"] = df_model["order_purchase_timestamp"].dt.hour

In [ ]:
# Por momento del dia
def time_of_day(hour):
    if 6 <= hour < 12:
        return "morning"
    elif 12 <= hour < 18:
        return "afternoon"
    elif 18 <= hour < 24:
        return "evening"
    else:
        return "night"

df_model["time_of_day"] = df_model["hour"].apply(time_of_day)


In [57]:
# chequeo final
print(df_model.shape)
print(df_model.head())
print(df_model.isnull().sum())

(118310, 19)
                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87   

   shipping_limit_date   price  freight_value  \
0  2017-09-19 09:45:35   58.90          13.29   
1  2017-05-03 11:05:13  239.90          19.93   
2  2018-01-18 14:48:30  199.00          17.87   
3  2018-08-15 10:10:18   12

DATASET FINAL

In [58]:
import os

PROCESSED_PATH = "../data/processed"
os.makedirs(PROCESSED_PATH, exist_ok=True)

In [59]:
df_model.to_csv(os.path.join(PROCESSED_PATH, "df_model.csv"), index=False)

## Definición de Macrogrupos de Categorías

Con el objetivo de simplificar la alta cardinalidad de categorías originales y facilitar tanto el análisis exploratorio como el modelado, se realizó una agrupación de las categorías en **macrogrupos**.

### Objetivo de la segmentación

- Reducir la complejidad del dataset  
- Agrupar productos según **lógica de consumo y comportamiento de compra**  
- Facilitar la detección de patrones para el sistema de recomendación  
- Permitir análisis más interpretables a nivel negocio  


### Criterio de agrupación

La división no se basó únicamente en similitud nominal, sino en:

- Tipo de uso del producto  
- Contexto de compra  
- Posibles relaciones de **cross-selling**  

Se definieron 4 macrogrupos principales:

#### 🏠 Home (Hogar y Decoración)
Incluye productos vinculados al hogar, mobiliario y confort.

Ejemplos:
- bed_bath_table  
- furniture_decor  
- housewares  
- home_comfort  


#### 💄 Personal Care (Cuidado Personal)
Productos relacionados con estética, accesorios personales y lifestyle.

Ejemplos:
- health_beauty  
- perfumery  
- fashion_bags_accessories  
- watches_gifts  


#### 💻 Tech (Tecnología y Entretenimiento)
Productos tecnológicos y de entretenimiento digital.

Ejemplos:
- computers_accessories  
- telephony  
- electronics  
- consoles_games  


#### 👶 Lifestyle (Familia, Mascotas y Recreación)
Productos asociados a actividades cotidianas, recreación y cuidado.

Ejemplos:
- baby  
- toys  
- pet_shop  
- sports_leisure  


### Implementación

La segmentación se realizó mediante un diccionario de mapeo aplicado sobre la variable:

`product_category_name_english`

```python
products["grupo_categoria"] = products["product_category_name_english"].map(mapa_grupos)

## **Manejo de valores nulos**

Durante el proceso de mapeo, pueden aparecer valores nulos debido a:

-Categorías que no fueron incluidas en el diccionario
-Registros con categoría original faltante

Para evitar pérdida de información y mantener consistencia, estos casos fueron asignados a una categoría residual:

.fillna("other")

Esto permite:
-No descartar datos relevantes
-Mantener integridad del dataset
-Facilitar análisis posteriores

## **CONCLUSION** ##
El análisis exploratorio de datos permitió comprender de manera integral la estructura, calidad y dinámica del dataset de comercio electrónico, evidenciando un sistema robusto compuesto por múltiples tablas interrelacionadas que describen el comportamiento de clientes, productos, pedidos, pagos y reseñas. Se identificó una alta calidad de la información, caracterizada por la ausencia de duplicados en las claves principales y una baja proporción de valores nulos, lo cual garantiza la confiabilidad de los resultados obtenidos y permite avanzar hacia etapas de modelado sin limitaciones significativas.